# Deep Learning 013 — Regression with Keras: Graduate Admissions

Companion notebook to the lesson. The third task type. Same network, three deliberate
changes:

| | Binary classification | Regression |
|---|---|---|
| output layer | `Dense(1, activation='sigmoid')` | **`Dense(1, activation='linear')`** |
| loss | `binary_crossentropy` | **`mean_squared_error`** |
| metric | accuracy | **$R^2$**, MAE, RMSE |

The target here — *Chance of Admit* — sits between 0 and 1, which makes it look like a
probability. It is not. It is a continuous quantity that happens to be bounded, and every
value in between is meaningful. **That is what makes it regression.**

> Fully runnable on scikit-learn; the Keras cell at the end is marked and optional.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/Admission_Predict_Ver1.1.csv")
df.columns = [c.strip() for c in df.columns]          # this file has trailing spaces
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
print(df.describe().T[["min", "max", "mean", "std"]].round(3).to_string())
print(f"\nmissing values: {int(df.isna().sum().sum())}")
print(f"duplicate rows: {int(df.duplicated().sum())}")

## Is this really regression?

The target runs 0.34 to 0.97. Three checks that it is a continuous quantity rather than a
class label:

In [ ]:
t = df["Chance of Admit"]
print(f"distinct values         : {t.nunique()} out of {len(t)} rows")
print(f"range                   : {t.min():.2f} to {t.max():.2f}")
print(f"is every value 0 or 1?  : {set(t.unique()) <= {0.0, 1.0}}")
print(f"\ntop 8 most common values: {sorted(t.value_counts().head(8).index.tolist())}")
print("\nA bounded, many-valued, ordered target where 0.71 means something different")
print("from 0.72. Regression.")

## Dropping the identifier, and choosing a scaler

`Serial No.` is a row index. It correlates with nothing and offers a network a way to
memorise the training set.

For scaling there is a real choice here, and the columns make it for you:

- **MinMax** when the bounds are known and fixed — GRE is out of 340, TOEFL out of 120,
  University Rating is 1–5.
- **Standard** when the distribution is roughly normal and the bounds are not meaningful.

CGPA is out of 10 and `Research` is already 0/1. MinMax puts everything on the same
footing without pretending any of it is Gaussian.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

X = df.drop(columns=["Serial No.", "Chance of Admit"])
y = df["Chance of Admit"].values
print("features:", X.columns.tolist())

X_tr, X_te, y_tr, y_te = train_test_split(X.values, y, test_size=0.2, random_state=0)
mm = MinMaxScaler().fit(X_tr)
X_tr_s, X_te_s = mm.transform(X_tr), mm.transform(X_te)
print(f"\ntrain {X_tr_s.shape}, test {X_te_s.shape}")
print(f"train range after MinMax: {X_tr_s.min():.3f} to {X_tr_s.max():.3f}")
print(f"test  range after MinMax: {X_te_s.min():.3f} to {X_te_s.max():.3f}")
print("\nper-column test maxima:", np.round(X_te_s.max(0), 3))

On this split the test set happens to stay inside [0, 1] — with 400 training rows covering
the full GRE and CGPA ranges, no test applicant is out of bounds. **But it is allowed to,
and that is the point.** The scaler learned its minimum and maximum from the training data
alone, so a future applicant with a higher GRE than anyone in training would legitimately
scale above 1. Fitting the scaler on all 500 rows would prevent that by construction — and
would quietly leak test information into training, making every score below slightly
optimistic.

## The baseline you must beat

$R^2 = 0$ does not mean "random". It means **exactly as good as always predicting the
mean**. Negative means worse than that. Compute the mean-predictor's score first so the
network's number has something to be compared against — and note which mean, because it
matters.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

for label, const in (("the TEST mean", y_te.mean()), ("the TRAIN mean", y_tr.mean())):
    pred = np.full_like(y_te, const)
    print(f"always predict {label} ({const:.4f}):")
    print(f"   R2 {r2_score(y_te, pred):>8.4f}"
          f"   MAE {mean_absolute_error(y_te, pred):>7.4f}"
          f"   RMSE {np.sqrt(mean_squared_error(y_te, pred)):>7.4f}")

Predicting the *test* mean scores exactly 0.0000, because that is the definition of
$R^2$. Predicting the *training* mean — the only one you are actually allowed to use —
scores **−0.0331**, slightly worse than zero, because the two means differ a little. A
negative $R^2$ is not always a broken model; sometimes it is an honest one.

## The network

Linear output. No activation on the last layer at all — the prediction is the weighted sum
itself, free to be any real number. Squashing it with a sigmoid would be an error even
though the target happens to live in [0, 1]: it would flatten the gradient near the ends,
exactly where several of the applicants are.

In [ ]:
from sklearn.neural_network import MLPRegressor

results = {}
for name, layers in (("no hidden layer (linear regression)", ()),
                     ("one hidden layer, 7 nodes", (7,)),
                     ("two hidden layers, 7 + 7", (7, 7))):
    if layers == ():
        from sklearn.linear_model import LinearRegression
        m = LinearRegression().fit(X_tr_s, y_tr)
    else:
        # tol and n_iter_no_change matter here: with the defaults this stops early
        # and scores 0.18, which looks like a modelling result and is not one.
        m = MLPRegressor(hidden_layer_sizes=layers, activation="relu", solver="adam",
                         max_iter=8000, random_state=0, learning_rate_init=0.01,
                         tol=1e-7, n_iter_no_change=200)
        m.fit(X_tr_s, y_tr)
    p = m.predict(X_te_s)
    results[name] = p
    print(f"{name:<38} R2 {r2_score(y_te, p):>7.4f}   MAE {mean_absolute_error(y_te, p):>7.4f}"
          f"   RMSE {np.sqrt(mean_squared_error(y_te, p)):>7.4f}")

Two things worth noticing, and the second is uncomfortable.

1. Every version beats the mean baseline by a wide margin — R² around 0.75 against 0. This
   data really is predictable.
2. **The network does not beat plain linear regression.** One hidden layer matches it to
   three decimals (0.766 against 0.766); two hidden layers are slightly *worse* at 0.738.
   That is not a failure of the notebook, it is what the data is like — seven roughly
   linear features and 500 rows. The honest reading is that a neural network is the wrong
   tool for this dataset. It is here because it is the smallest complete worked example of
   the regression *setup*, not because it wins, and a lesson that only ever showed you
   cases where the network wins would be teaching you the wrong instinct.

One practical note buried in that cell: with scikit-learn's default `tol` and
`n_iter_no_change`, the single-hidden-layer model stops early and scores **0.18**. That
number looks like a finding about model capacity and is nothing of the kind — it is an
optimiser giving up. Always check `n_iter_` before drawing a conclusion from a bad score.

In [ ]:
# where does the error actually live?
p = results["one hidden layer, 7 nodes"]
err = y_te - p
order = np.argsort(y_te)
print(f"{'true chance':>13}{'n':>5}{'mean abs error':>17}")
for lo, hi in ((0.30, 0.60), (0.60, 0.75), (0.75, 0.90), (0.90, 1.01)):
    m_ = (y_te >= lo) & (y_te < hi)
    if m_.sum():
        print(f"  {lo:.2f} - {hi:.2f}{m_.sum():>7}{np.abs(err[m_]).mean():>17.4f}")
print(f"\nworst single prediction: true {y_te[np.abs(err).argmax()]:.2f}, "
      f"predicted {p[np.abs(err).argmax()]:.2f}")
print(f"predictions outside [0, 1]: {int(((p < 0) | (p > 1)).sum())}")

The error is not spread evenly — the low-chance applicants are the hard ones, and there are
few of them to learn from. A single aggregate score hides that completely, which is the
argument for looking at the residuals grouped by target value rather than trusting $R^2$
alone.

Note also whether any predictions escaped [0, 1]. A linear output has no idea the target is
bounded, so it can. Clipping afterwards is a legitimate, and honest, post-processing step —
it is not the same as putting a sigmoid on the output.

## Which features matter

In [ ]:
corr = df.drop(columns=["Serial No."]).corr()["Chance of Admit"].drop("Chance of Admit")
print(corr.sort_values(ascending=False).round(3).to_string())
print("\nCGPA dominates, which matches every admissions officer's intuition and is")
print("a useful sanity check that nothing has been shuffled or mis-joined.")

## The Keras version

In [ ]:
# --- needs TensorFlow ---
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Input(shape=(X_tr_s.shape[1],)),
    keras.layers.Dense(7, activation="relu"),
    keras.layers.Dense(1, activation="linear"),      # <- the whole difference
])
model.compile(optimizer="adam", loss="mean_squared_error", metrics=["mae"])
model.summary()

history = model.fit(X_tr_s, y_tr, epochs=100, batch_size=32,
                    validation_split=0.2, verbose=0)
pred = model.predict(X_te_s, verbose=0).ravel()
print(f"R2 {r2_score(y_te, pred):.4f}   MAE {mean_absolute_error(y_te, pred):.4f}")

The lesson reports reaching **R² ≈ 0.76** with a hidden layer and around 100 epochs. The
scikit-learn runs above land in the same region, which is the useful check — two different
libraries, same data, comparable score.

## Diagnosing with a plot

```python
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set(xlabel="epoch", ylabel="MSE"); ax[0].legend()

ax[1].scatter(y_te, pred, s=14)
ax[1].plot([0.3, 1], [0.3, 1], "k--")       # perfect prediction
ax[1].set(xlabel="true", ylabel="predicted")
```

The scatter is the more informative of the two. Points on the dashed line are perfect; a
systematic bend away from it means the model is biased in that region, which no single
metric will tell you.

## Try it yourself

1. Swap `MinMaxScaler` for `StandardScaler`. Does $R^2$ move? Should it, for a network?
2. Put a sigmoid on the output layer instead of a linear activation and retrain. The target
   is in [0, 1], so it *should* work — measure how much worse it is and explain why.
3. Drop CGPA and retrain. How much of the model's skill was that one column?
4. Compute $R^2$ on the training set as well. A large gap between train and test $R^2$ is
   overfitting; with 400 training rows and a 7-node hidden layer, is there one?